<a href="https://colab.research.google.com/github/nexageapps/AI/blob/main/Basic/B07%20-%20Model%20Evaluation%20and%20Performance%20Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7: Model Evaluation and Performance Metrics

**Interactive Application:** [Model Evaluation Dashboard](https://nexageapps.github.io/AI/compsci713/week3/model-evaluation/) - Practice these concepts hands-on!

## Learning Objectives
- Understand why accuracy alone is misleading
- Master classification metrics (Precision, Recall, F1, ROC-AUC)
- Learn regression metrics (MAE, MSE, RMSE, R²)
- Implement cross-validation techniques
- Handle imbalanced datasets effectively
- Interpret confusion matrices and ROC curves

## Why This Matters for MAI Students

Imagine a model that predicts 99% accuracy - sounds great, right? But what if:
- It's detecting cancer, and the 1% it misses are all actual cancer cases?
- It's fraud detection, and it flags 99% of legitimate transactions as fraud?

**Accuracy alone can be dangerously misleading.** This lesson teaches you how to properly evaluate models based on what matters for your specific problem.

## Real-World Applications
- **Medical Diagnosis**: Minimize false negatives (missing diseases)
- **Fraud Detection**: Balance false positives vs false negatives
- **Spam Filtering**: Precision-recall trade-off
- **Credit Scoring**: Regulatory compliance and fairness
- **Autonomous Vehicles**: Safety-critical decision making

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.utils import resample

sns.set_style('whitegrid')
print("Libraries imported successfully")
print("="*80)

## Part 1: The Problem with Accuracy

### Example: Cancer Detection

Let's say we have a dataset where only 1% of patients have cancer. A naive model that always predicts "No Cancer" would achieve 99% accuracy - but it would be completely useless!

This is why we need better metrics.

In [ ]:
# Simulate imbalanced cancer detection dataset
np.random.seed(42)

# 1000 patients, only 10 have cancer (1%)
n_samples = 1000
n_cancer = 10

y_true = np.array([1]*n_cancer + [0]*(n_samples - n_cancer))
np.random.shuffle(y_true)

# Naive model: always predict "No Cancer" (0)
y_pred_naive = np.zeros(n_samples)

# Calculate accuracy
accuracy_naive = accuracy_score(y_true, y_pred_naive)

print("Naive Model (Always predicts 'No Cancer'):")
print(f"Accuracy: {accuracy_naive:.2%}")
print(f"\nBut this model missed ALL {n_cancer} cancer cases!")
print("This is why accuracy is misleading for imbalanced data.")

## Part 2: Classification Metrics Deep Dive

### Understanding the Confusion Matrix

The confusion matrix is the foundation of all classification metrics:

```
                    Predicted
                 Negative  Positive
Actual Negative     TN        FP
Actual Positive     FN        TP
```

- **True Positive (TP)**: Correctly predicted positive
- **True Negative (TN)**: Correctly predicted negative
- **False Positive (FP)**: Incorrectly predicted positive (Type I error)
- **False Negative (FN)**: Incorrectly predicted negative (Type II error)

### Key Metrics:

1. **Precision** = TP / (TP + FP)
   - "Of all positive predictions, how many were correct?"
   - Important when false positives are costly

2. **Recall (Sensitivity)** = TP / (TP + FN)
   - "Of all actual positives, how many did we find?"
   - Important when false negatives are costly

3. **F1-Score** = 2 × (Precision × Recall) / (Precision + Recall)
   - Harmonic mean of precision and recall
   - Useful when you need balance

4. **Specificity** = TN / (TN + FP)
   - "Of all actual negatives, how many did we correctly identify?"

In [ ]:
# Create a realistic imbalanced classification problem
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    weights=[0.9, 0.1],  # 90% class 0, 10% class 1
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train a logistic regression model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Model trained on imbalanced data")
print(f"Training set: {len(y_train)} samples")
print(f"Test set: {len(y_test)} samples")
print(f"Class distribution: {np.bincount(y_test)}")

In [ ]:
# Calculate all classification metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Classification Metrics:")
print("="*50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}  (Of predicted positives, how many correct?)")
print(f"Recall:    {recall:.4f}  (Of actual positives, how many found?)")
print(f"F1-Score:  {f1:.4f}  (Harmonic mean of precision and recall)")
print("\n" + "="*50)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives:  {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives:  {cm[1,1]}")

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

# Print detailed classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

## Part 3: ROC Curve and AUC

### What is ROC?

**ROC (Receiver Operating Characteristic)** curve shows the trade-off between:
- **True Positive Rate (TPR)** = Recall = TP / (TP + FN)
- **False Positive Rate (FPR)** = FP / (FP + TN)

### What is AUC?

**AUC (Area Under the Curve)** measures the entire area under the ROC curve:
- AUC = 1.0: Perfect classifier
- AUC = 0.5: Random classifier (no better than coin flip)
- AUC < 0.5: Worse than random (predictions are inverted)

### Why ROC-AUC is Important:
- **Threshold-independent**: Evaluates model across all possible thresholds
- **Handles imbalance**: Works well with imbalanced datasets
- **Probabilistic**: Uses predicted probabilities, not just binary predictions

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Plot ROC curve
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, 
         label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', 
         label='Random Classifier (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print(f"\nROC-AUC Score: {roc_auc:.4f}")
print("\nInterpretation:")
if roc_auc > 0.9:
    print("Excellent model performance")
elif roc_auc > 0.8:
    print("Good model performance")
elif roc_auc > 0.7:
    print("Fair model performance")
else:
    print("Poor model performance")

## Part 4: Regression Metrics

For regression problems, we use different metrics:

### 1. Mean Absolute Error (MAE)
- Average absolute difference between predictions and actual values
- **Formula**: MAE = (1/n) × Σ|y_true - y_pred|
- **Interpretation**: Same units as target variable
- **Robust**: Less sensitive to outliers

### 2. Mean Squared Error (MSE)
- Average squared difference between predictions and actual values
- **Formula**: MSE = (1/n) × Σ(y_true - y_pred)²
- **Interpretation**: Squared units (harder to interpret)
- **Sensitive**: Heavily penalizes large errors

### 3. Root Mean Squared Error (RMSE)
- Square root of MSE
- **Formula**: RMSE = √MSE
- **Interpretation**: Same units as target variable
- **Popular**: Most commonly used regression metric

### 4. R² Score (Coefficient of Determination)
- Proportion of variance explained by the model
- **Formula**: R² = 1 - (SS_res / SS_tot)
- **Range**: -∞ to 1.0
- **Interpretation**:
  - R² = 1.0: Perfect predictions
  - R² = 0.0: Model as good as predicting mean
  - R² < 0.0: Model worse than predicting mean

In [ ]:
# Create regression dataset
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=10,
    n_informative=8,
    noise=10,
    random_state=42
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

# Train linear regression model
reg_model = LinearRegression()
reg_model.fit(X_train_reg, y_train_reg)
y_pred_reg = reg_model.predict(X_test_reg)

# Calculate regression metrics
mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print("Regression Metrics:")
print("="*50)
print(f"MAE (Mean Absolute Error):       {mae:.4f}")
print(f"MSE (Mean Squared Error):        {mse:.4f}")
print(f"RMSE (Root Mean Squared Error):  {rmse:.4f}")
print(f"R² Score:                        {r2:.4f}")
print("\n" + "="*50)
print(f"\nInterpretation: Model explains {r2*100:.2f}% of variance in target")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(12, 5))

# Plot 1: Predictions vs Actual
plt.subplot(1, 2, 1)
plt.scatter(y_test_reg, y_pred_reg, alpha=0.5)
plt.plot([y_test_reg.min(), y_test_reg.max()], 
         [y_test_reg.min(), y_test_reg.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Predictions vs Actual Values')
plt.legend()
plt.grid(alpha=0.3)

# Plot 2: Residuals
plt.subplot(1, 2, 2)
residuals = y_test_reg - y_pred_reg
plt.scatter(y_pred_reg, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 5: Cross-Validation

### Why Cross-Validation?

A single train-test split can be misleading:
- Results depend on which data points end up in train vs test
- May get lucky (or unlucky) with the split
- Doesn't use all data for both training and testing

### K-Fold Cross-Validation

1. Split data into K equal folds
2. Train on K-1 folds, test on remaining fold
3. Repeat K times, each fold used as test set once
4. Average the K results

### Stratified K-Fold

For classification with imbalanced classes:
- Ensures each fold has same class distribution as original dataset
- Prevents folds with too few examples of minority class

In [ ]:
# Standard K-Fold Cross-Validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')

print("5-Fold Cross-Validation Results:")
print("="*50)
for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.4f}")
print("="*50)
print(f"Mean Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print(f"\nThis gives us confidence interval for model performance")

In [ ]:
# Stratified K-Fold (better for imbalanced data)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratified_scores = cross_val_score(model, X, y, cv=stratified_kfold, scoring='f1')

print("\nStratified 5-Fold Cross-Validation (F1 Score):")
print("="*50)
for i, score in enumerate(stratified_scores, 1):
    print(f"Fold {i}: {score:.4f}")
print("="*50)
print(f"Mean F1 Score: {stratified_scores.mean():.4f} (+/- {stratified_scores.std() * 2:.4f})")

# Visualize cross-validation scores
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.bar(range(1, 6), cv_scores)
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label='Mean')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('K-Fold Cross-Validation Scores')
plt.legend()
plt.ylim([0, 1])

plt.subplot(1, 2, 2)
plt.bar(range(1, 6), stratified_scores)
plt.axhline(y=stratified_scores.mean(), color='r', linestyle='--', label='Mean')
plt.xlabel('Fold')
plt.ylabel('F1 Score')
plt.title('Stratified K-Fold CV Scores')
plt.legend()
plt.ylim([0, 1])

plt.tight_layout()
plt.show()

## Part 6: Handling Imbalanced Datasets

### Techniques for Imbalanced Data:

1. **Class Weights**: Penalize misclassification of minority class more
2. **Oversampling**: Duplicate minority class samples
3. **Undersampling**: Remove majority class samples
4. **SMOTE**: Synthetic Minority Over-sampling Technique
5. **Ensemble Methods**: Use algorithms designed for imbalance

### Choosing the Right Metric:

- **Balanced data**: Accuracy is fine
- **Imbalanced data**: Use F1, Precision, Recall, or ROC-AUC
- **Cost-sensitive**: Choose metric based on business cost of errors

In [ ]:
# Create highly imbalanced dataset
X_imb, y_imb = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    weights=[0.95, 0.05],  # 95% class 0, 5% class 1
    random_state=42
)

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb
)

print("Imbalanced Dataset:")
print(f"Training set class distribution: {np.bincount(y_train_imb)}")
print(f"Test set class distribution: {np.bincount(y_test_imb)}")
print(f"Imbalance ratio: {np.bincount(y_train_imb)[0] / np.bincount(y_train_imb)[1]:.1f}:1")

In [ ]:
# Compare models with and without class weights
model_no_weights = LogisticRegression(random_state=42, max_iter=1000)
model_with_weights = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)

model_no_weights.fit(X_train_imb, y_train_imb)
model_with_weights.fit(X_train_imb, y_train_imb)

y_pred_no_weights = model_no_weights.predict(X_test_imb)
y_pred_with_weights = model_with_weights.predict(X_test_imb)

print("Model WITHOUT Class Weights:")
print("="*50)
print(f"Accuracy:  {accuracy_score(y_test_imb, y_pred_no_weights):.4f}")
print(f"Precision: {precision_score(y_test_imb, y_pred_no_weights):.4f}")
print(f"Recall:    {recall_score(y_test_imb, y_pred_no_weights):.4f}")
print(f"F1-Score:  {f1_score(y_test_imb, y_pred_no_weights):.4f}")

print("\nModel WITH Class Weights (Balanced):")
print("="*50)
print(f"Accuracy:  {accuracy_score(y_test_imb, y_pred_with_weights):.4f}")
print(f"Precision: {precision_score(y_test_imb, y_pred_with_weights):.4f}")
print(f"Recall:    {recall_score(y_test_imb, y_pred_with_weights):.4f}")
print(f"F1-Score:  {f1_score(y_test_imb, y_pred_with_weights):.4f}")

print("\nNotice: Class weights improve recall (finding minority class)")
print("Trade-off: Slightly lower precision, but better overall F1")

## Part 7: Choosing the Right Metric

### Decision Guide:

| Problem Type | Recommended Metric | When to Use |
|--------------|-------------------|-------------|
| Balanced Classification | Accuracy | Classes roughly equal |
| Imbalanced Classification | F1-Score | Balance precision and recall |
| Medical Diagnosis | Recall | Minimize false negatives |
| Spam Detection | Precision | Minimize false positives |
| Ranking/Probability | ROC-AUC | Threshold-independent evaluation |
| Regression | RMSE | Standard regression metric |
| Regression with Outliers | MAE | Robust to outliers |
| Variance Explained | R² | How well model fits data |

### Real-World Examples:

1. **Cancer Detection**: Maximize Recall
   - Missing cancer (false negative) is catastrophic
   - False alarm (false positive) is acceptable

2. **Fraud Detection**: Balance Precision and Recall
   - False positives annoy customers
   - False negatives cost money
   - Use F1-Score

3. **Spam Filter**: Maximize Precision
   - False positive (legitimate email marked spam) is very bad
   - False negative (spam in inbox) is tolerable

4. **Credit Scoring**: Use ROC-AUC
   - Need to rank applicants by risk
   - Threshold can be adjusted based on business needs

## Key Takeaways

1. **Accuracy is Not Enough**
   - Can be misleading with imbalanced data
   - Always look at confusion matrix

2. **Understand the Trade-offs**
   - Precision vs Recall
   - False positives vs False negatives
   - Choose metric based on business cost

3. **Use Cross-Validation**
   - Single train-test split can be misleading
   - K-fold gives confidence intervals
   - Stratified K-fold for imbalanced data

4. **Handle Imbalanced Data**
   - Use class weights
   - Consider resampling techniques
   - Choose appropriate metrics (F1, ROC-AUC)

5. **Regression Metrics Matter**
   - MAE for interpretability
   - RMSE for standard evaluation
   - R² for variance explained

6. **Context is Everything**
   - Medical: Minimize false negatives
   - Spam: Minimize false positives
   - Fraud: Balance both

## Next Steps

- **B8**: Learn about overfitting and regularization techniques
- **B9**: Apply these metrics to CNNs for image classification
- **Practice**: Evaluate models on real-world imbalanced datasets

## References

1. Scikit-learn Metrics Documentation: https://scikit-learn.org/stable/modules/model_evaluation.html
2. "Imbalanced Learning: Foundations, Algorithms, and Applications" by He & Ma
3. "Evaluating Machine Learning Models" by Alice Zheng
4. ROC Analysis: Fawcett, T. (2006). "An introduction to ROC analysis"

---

**Remember**: The best metric depends on your specific problem and the cost of different types of errors. Always consider the real-world implications of your model's mistakes!

---

## Exercises

Test your understanding with these hands-on exercises. Try to solve them before looking at the hints.


### Exercise 1: Confusion Matrix Analysis

Given the following predictions from a medical diagnosis model, compute the confusion matrix, then calculate precision, recall, F1-score, and accuracy **by hand** (using Python but not sklearn).

```
y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1]
y_pred = [1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0]
```

Then verify your results using `sklearn.metrics`.



In [ ]:
import numpy as np

y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1]
y_pred = [1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0]

# Compute TP, FP, TN, FN manually
# TP = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
# ... compute FP, TN, FN

# Calculate metrics
# precision = TP / (TP + FP)
# recall = TP / (TP + FN)
# f1 = 2 * precision * recall / (precision + recall)

# Your code here


### Exercise 2: ROC Curve and Threshold Selection

Train a binary classifier on a synthetic imbalanced dataset (90% class 0, 10% class 1). Plot the ROC curve and find the optimal threshold that maximizes the F1-score (not just accuracy). Show how different thresholds affect precision and recall.



In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Create imbalanced dataset
# np.random.seed(42)
# X = np.random.randn(1000, 5)
# y = (np.random.rand(1000) < 0.1).astype(int)  # 10% positive

# Train model, get prediction probabilities
# Try different thresholds: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
# For each threshold, compute precision, recall, F1

# Your code here


---

## Key Takeaways

**Relevant UoA Courses:** COMPSCI 762, COMPSCI 761

1. Accuracy = (TP + TN) / Total - can be misleading with imbalanced data
2. Precision = TP / (TP + FP) - of predicted positives, how many are correct?
3. Recall = TP / (TP + FN) - of actual positives, how many did we find?
4. F1 Score = 2·(Precision·Recall)/(Precision+Recall) - harmonic mean
5. Cross-validation: split data into k folds, train on k-1, test on 1, repeat

---

## Exam Preparation Guide

### Essential Concepts for Exams

- Calculate metrics from confusion matrix: [[TN, FP], [FN, TP]]
- Know when to prioritize precision vs recall (medical: recall, spam: precision)
- Understand ROC curve: TPR vs FPR at different thresholds
- Explain why accuracy fails with imbalanced data (99% negative → 99% accuracy by predicting all negative)
- Common question: Given confusion matrix, calculate all metrics

### Common Mistakes to Avoid

- ❌ Confusing precision and recall
- ❌ Using accuracy for imbalanced datasets
- ❌ Not understanding trade-off between precision and recall

### Practice Problems

1. Confusion matrix: [[90, 10], [5, 95]]. Calculate accuracy, precision, recall, F1
2. Dataset: 95% negative, 5% positive. Model predicts all negative. What's accuracy?
3. When would you prefer high recall over high precision?

### How This Helps Your UoA Courses

**COMPSCI 762, COMPSCI 761:**
- Provides hands-on implementation of theoretical concepts
- Practice problems similar to exam questions
- Reinforces lecture material with code examples
- Helps build intuition for complex topics

### Study Tips

1. **Understand, Don't Memorize**: Focus on why, not just what
2. **Practice Calculations**: Work through problems by hand
3. **Connect to Theory**: Link code to mathematical formulations
4. **Teach Others**: Explaining concepts solidifies understanding
5. **Review Regularly**: Spaced repetition improves retention

### Exam Question Types

- **Conceptual**: Explain why/how something works
- **Calculation**: Compute outputs, gradients, shapes
- **Comparison**: Compare approaches, trade-offs
- **Application**: Design solution for given problem
- **Debugging**: Identify and fix issues

---


---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (B07)
- [ ] Performance metrics (accuracy, precision, recall, F1)
- [ ] Cross-validation techniques
- [ ] Confusion matrix
- [ ] ROC curves and AUC

---